# Motorforsikring: innlasting, datarensing og kvalitetskontroll

Notebooken viser resultatene fra datakvalitetsmodulen. Variabelordbok,
rensing og integritetsdiagnostikk ligger i `src/data_quality.py`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.data_quality import (
    build_integrity_diagnostics,
    build_variable_dictionary,
    clean_motor_data,
    find_variables,
    run_integrity_checks,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)
DATA_PATH = Path("data/Dataset of motor insurance portfolio.csv")
DESCRIPTION_PATH = Path("data/Descriptive of variables.xlsx")
if not DATA_PATH.exists() or not DESCRIPTION_PATH.exists():
    raise FileNotFoundError("Kjør notebooken fra prosjektroten slik at begge filene i data/ er tilgjengelige.")

## 1. Uendret innlasting og første inspeksjon

CSV-en leses før rensing. Excel-filen brukes som kildekontroll, mens den
forbedrede variabelordboken bygges i scriptet.

In [ ]:
raw_data = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8", low_memory=False)
source_dictionary_raw = pd.read_excel(DESCRIPTION_PATH, sheet_name="Cartera", usecols=["Variables", "Description", "Group"])
source_variables = source_dictionary_raw["Variables"].dropna().astype(str).tolist()
variable_dictionary = build_variable_dictionary()
raw_overview = pd.Series({
    "Antall rader": len(raw_data),
    "Antall kolonner": raw_data.shape[1],
    "Unike insured_id": raw_data["insured_id"].nunique(),
    "Duplikate hele rader": int(raw_data.duplicated().sum()),
    "Duplikate (insured_id, year)": int(raw_data.duplicated(["insured_id", "year"]).sum()),
    "Minste år": int(raw_data["year"].min()),
    "Største år": int(raw_data["year"].max()),
    "Variabler dokumentert i Excel": len(source_variables),
    "Excel- og CSV-variabler er identiske": source_variables == raw_data.columns.tolist(),
    "Minnestørrelse (MiB)": round(raw_data.memory_usage(deep=True).sum() / 1024**2, 1),
}, name="Verdi").to_frame()
display(raw_overview)
display(raw_data.head())

,Verdi
Antall rader,354140
Antall kolonner,47
Unike insured_id,185678
Duplikate hele rader,0
"Duplikate (insured_id, year)",0
Minste år,2022
Største år,2024
Variabler dokumentert i Excel,47
Excel- og CSV-variabler er identiske,True
Minnestørrelse (MiB),257.4


,insured_id,year,policy_type,policy_status,business_type,payment_frequency,bonus_score,driver_age,vehicle_age,age_driving_licence,fuel_type,vehicle_value,seats,power_to_weight_ratio,vehicle_brand,municipality_type,circulation_area,total_premium,liability_premium,property_damage_premium,theft_premium,fire_premium,glass_premium,legal_protection_premium,occupants_premium,total_claims,liability_claims,liability_property_claims,liability_injury_claims,property_claims,theft_claims,fire_claims,glass_claims,legal_protection_claims,occupants_claims,total_incurred,liability_incurred,liability_property_incurred,liability_injury_incurred,property_incurred,theft_incurred,fire_incurred,glass_incurred,legal_protection_incurred,occupants_incurred,total_exposure,liability_exposure
0,1,2022,COMP_E,C,NB,A,G,48,22.0,26.0,G,149511.4425,4,3.92,PORSCHE,I,U,279.557558,30.602324,121.912690,112.361489,3.832441,8.305788,2.043622,0.499205,0,0,0,0,0,0,0,0,0,0,0.0000,0.000,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.09863,0.09863
1,2,2022,COMP_E,A,P,A,G,43,24.0,19.0,D,65065.1232,8,10.15,MERCEDES,I,U,546.623415,114.333356,303.105967,80.784824,7.826861,30.091751,7.812747,2.667908,0,0,0,0,0,0,0,0,0,0,0.0000,0.000,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
2,2,2023,COMP_E,A,P,A,G,44,25.0,19.0,D,65065.1232,8,10.15,MERCEDES,I,U,548.752558,110.666891,304.955990,83.521384,6.151465,32.861626,7.655637,2.939566,1,1,1,0,0,0,0,0,0,0,233.0130,233.013,233.013,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
3,2,2024,COMP_E,A,P,A,G,45,26.0,18.0,D,65065.1232,8,10.15,MERCEDES,I,U,584.632430,122.092532,305.225093,107.886872,5.842892,31.101584,10.084173,2.399283,2,1,1,0,1,0,0,0,0,0,2586.3615,1035.000,1035.000,0.0,1551.3615,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
4,3,2022,COMP_E,A,NB,S,G,45,26.0,19.0,D,51626.6625,7,9.19,KIA,I,R,687.490508,214.583989,354.874140,53.077620,7.715765,48.574298,6.700701,1.963995,0,0,0,0,0,0,0,0,0,0,0.0000,0.000,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000


In [ ]:
raw_schema = pd.DataFrame({
    "raw_dtype": raw_data.dtypes.astype(str),
    "missing_count": raw_data.isna().sum(),
    "missing_percent": raw_data.isna().mean().mul(100).round(4),
    "unique_count": raw_data.nunique(dropna=False),
    "example": [raw_data[column].dropna().iloc[0] if raw_data[column].notna().any() else pd.NA for column in raw_data.columns],
})
display(raw_schema)
display(raw_data.isna().groupby(raw_data["year"]).sum().loc[:, lambda frame: frame.sum().gt(0)].T)

,raw_dtype,missing_count,missing_percent,unique_count,example
insured_id,int64,0,0.0000,185678,1
year,int64,0,0.0000,3,2022
policy_type,str,0,0.0000,5,COMP_E
policy_status,str,0,0.0000,2,C
business_type,str,0,0.0000,2,NB
payment_frequency,str,0,0.0000,3,A
bonus_score,str,0,0.0000,3,G
driver_age,int64,0,0.0000,73,48
vehicle_age,float64,2,0.0006,69,22.0
age_driving_licence,float64,2,0.0006,72,26.0


year,2022,2023,2024
vehicle_age,0,1,1
age_driving_licence,0,1,1
fuel_type,171,397,719
vehicle_value,111,173,229


## 2. Variabelordbok

`variable_dictionary` er en dataframe bygget i scriptet og kan filtreres
senere med `find_variables(variable_dictionary, ...)`.

In [ ]:
display(find_variables(variable_dictionary, group="Skadeantall"))
display(find_variables(variable_dictionary, search="ansvar"))

,variable,group,description,unit_or_codes,analysis_role,clean_dtype,missing_allowed
0,total_claims,Skadeantall,Totalt antall meldte skader på tvers av dekninger i eksponeringsperioden.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
1,liability_claims,Skadeantall,Totalt antall ansvarsskader; sum av materielle skader og personskader under ansvar.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
2,liability_property_claims,Skadeantall,Antall ansvarsskader med materiell skade på tredjepart.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
3,liability_injury_claims,Skadeantall,Antall ansvarsskader med personskade på tredjepart.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
4,property_claims,Skadeantall,Antall skader på eget kjøretøy under kaskodekning.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
5,theft_claims,Skadeantall,Antall tyveriskader.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
6,fire_claims,Skadeantall,Antall brannskader.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
7,glass_claims,Skadeantall,Antall glasskader.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
8,legal_protection_claims,Skadeantall,Antall rettshjelpsskader/-saker.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
9,occupants_claims,Skadeantall,Antall skader under passasjerdekningen.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei


,variable,group,description,unit_or_codes,analysis_role,clean_dtype,missing_allowed
0,policy_type,Polise og kontrakt,Overordnet produkt-/dekningsstruktur for polisen.,TP=ansvar; TPG=ansvar+glass; CC=ansvar+minst to tilleggsdekninger; COMP_E=kasko med egenandel; COMP_N=kasko uten ege...,Kategorisk risikofaktor,category,Nei
1,liability_premium,Premier,Nettopremie for ansvarsforsikring.,Beløp (artikkelen presenterer premier i EUR); null betyr normalt at dekningen ikke er tegnet eller at eksponeringen ...,Premie,float64,Nei
2,liability_claims,Skadeantall,Totalt antall ansvarsskader; sum av materielle skader og personskader under ansvar.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
3,liability_property_claims,Skadeantall,Antall ansvarsskader med materiell skade på tredjepart.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
4,liability_injury_claims,Skadeantall,Antall ansvarsskader med personskade på tredjepart.,Ikke-negativt heltallsantall,Respons-/utfallsvariabel,Int16,Nei
5,liability_incurred,Påløpt skadekostnad,Påløpt kostnad for ansvarsskader; sum av materiell skade og personskade under ansvar.,Beløp (antatt EUR); betalt beløp pluss utestående reserve,Respons-/utfallsvariabel,float64,Nei
6,liability_property_incurred,Påløpt skadekostnad,Påløpt kostnad for materielle ansvarsskader.,Beløp (antatt EUR); betalt beløp pluss utestående reserve,Respons-/utfallsvariabel,float64,Nei
7,liability_injury_incurred,Påløpt skadekostnad,Påløpt kostnad for personskader under ansvar.,Beløp (antatt EUR); betalt beløp pluss utestående reserve,Respons-/utfallsvariabel,float64,Nei
8,liability_exposure,Eksponering,Eksponeringstid spesifikt for ansvarsdekningen.,"Poliseår i intervallet [0, 1]",Eksponering/offset,float64,Nei


## 3. Rensing

Rensingen låser skjema og datatyper, standardiserer kategorier og gjør kun den
dokumenterte endringen av ikke-positive `power_to_weight_ratio`-verdier.

In [ ]:
data, cleaning_log = clean_motor_data(raw_data, variable_dictionary)
display(cleaning_log)
display(pd.DataFrame({"raw_dtype": raw_data.dtypes.astype(str), "clean_dtype": data.dtypes.astype(str), "missing_after_cleaning": data.isna().sum()}))

,step,affected_rows,result
0,"Kontroll av skjema, kategorikoder og unik poliseår-nøkkel",0,Bestått
1,Fjerning av omkringliggende mellomrom i tekstfelt,0,Standardisert
2,Ikke-positiv power_to_weight_ratio satt til manglende,3,Standardisert
3,Rader slettet eller imputert,0,Ingen


,raw_dtype,clean_dtype,missing_after_cleaning
insured_id,int64,Int32,0
year,int64,Int16,0
policy_type,str,category,0
policy_status,str,category,0
business_type,str,category,0
payment_frequency,str,category,0
bonus_score,str,category,0
driver_age,int64,Int16,0
vehicle_age,float64,Int16,2
age_driving_licence,float64,Int16,2


## 4. Integritetsdiagnostikk

Alle kontroller kjøres i scriptet. Notebooken presenterer oppsummeringene og
beholder dem som dataframes/datastrukturer for senere oppslag.

In [ ]:
integrity_checks = run_integrity_checks(data, variable_dictionary)
diagnostics = build_integrity_diagnostics(data, integrity_checks)
critical_failures = diagnostics["critical_failures"]
warning_examples = diagnostics["warning_examples"]
coverage_diagnostics = diagnostics["coverage_diagnostics"]
temporal_diagnostics = diagnostics["temporal_diagnostics"]
display(integrity_checks)
display(warning_examples)
display(coverage_diagnostics)
display(temporal_diagnostics)
if not critical_failures.empty:
    raise AssertionError("Minst én kritisk integritetskontroll feilet. Se critical_failures.")

,check,severity,expectation,violating_rows,status
0,"Unik (insured_id, year)",Kritisk,Ingen duplikate poliseår,0,Bestått
1,Manglende i obligatoriske felt,Kritisk,Ingen manglende verdier,0,Bestått
2,Endelige numeriske verdier,Kritisk,Ingen +inf eller -inf,0,Bestått
3,Ikke-negative premier,Kritisk,Alle premier >= 0,0,Bestått
4,Ikke-negative skadeantall,Kritisk,Alle skadeantall >= 0,0,Bestått
5,Ikke-negative incurred-beløp,Kritisk,Alle incurred-beløp >= 0,0,Bestått
6,Eksponering innenfor gyldig område,Kritisk,"Begge eksponeringer i [0, 1]",0,Bestått
7,Totalpremie summerer,Kritisk,total_premium = sum dekningspremier,0,Bestått
8,Totalt skadeantall summerer,Kritisk,total_claims = sum skadeantall per dekning,0,Bestått
9,Ansvarsskadeantall summerer,Kritisk,liability_claims = property + injury under ansvar,0,Bestått


,check,insured_id,year,policy_type,policy_status,business_type,age_driving_licence,liability_premium,total_premium,total_claims,total_incurred,total_exposure
0,Kansellert polise med full eksponering,8053,2022,TPG,C,NB,19,549.605184,596.863036,2,304.8305,1.000000
1,Positiv ansvarseksponering med null ansvarspremie,50075,2022,CC,A,NB,20,0.000000,44.522410,0,0.0000,0.539726
2,Positiv ansvarseksponering med null ansvarspremie,73579,2023,COMP_E,A,NB,20,0.000000,107.349196,0,0.0000,0.369863
3,Positiv ansvarseksponering med null ansvarspremie,73579,2024,COMP_E,A,NB,19,0.000000,294.780318,0,0.0000,1.000000
4,Positiv ansvarseksponering med null ansvarspremie,78590,2023,COMP_E,A,NB,19,0.000000,269.079700,0,0.0000,0.813699
5,Positiv ansvarseksponering med null ansvarspremie,78590,2024,COMP_E,A,NB,19,0.000000,326.888858,0,0.0000,1.000000
6,Skade eller incurred ved null eksponering,23744,2022,CC,C,NB,40,0.000000,0.000000,1,180.1130,0.000000
7,Registrert skade med null incurred,14,2024,CC,A,P,19,220.026475,295.208262,1,0.0000,1.000000
8,Registrert skade med null incurred,65,2023,COMP_E,A,P,22,128.977073,411.714008,1,0.0000,1.000000
9,Registrert skade med null incurred,66,2022,COMP_N,A,NB,19,170.171205,960.735636,1,0.0000,1.000000


,coverage,zero_premium_rows,claim_with_zero_premium,incurred_with_zero_premium
0,Ansvar,72,1,1
1,Egen skade,244802,2,2
2,Tyveri,42180,0,0
3,Brann,42183,0,0
4,Glass,5839,0,0
5,Rettshjelp,67,0,0
6,Passasjer,69,0,0


,variable,comparable_transitions,unchanged,decrease,increase_larger_than_year_gap
0,driver_age,168462,54527,42,14
1,vehicle_age,168461,54494,113,77
2,age_driving_licence,168461,147969,19571,57


## 5. Renseutfall og avgrensning for neste fase

Det rensede analysegrunnlaget ligger i `data`. Ingen rader er slettet og ingen
verdier er imputert. Diagnosene er funn som må vurderes før modellering.